# Alternative Spatial Testing Projection

Curated from the archived research notebook `4. SSH projection data(testing)-2D.ipynb`. Read the repository
README and `docs/limitations.md` before execution. All original files and
execution outputs were preserved separately.

Full preprocessing requires input files and an `internal_waves` module that
were not included in the available collection. See `docs/reproduction.md`.


In [ ]:
from pathlib import Path
import os
import sys

start = Path.cwd().resolve()
candidates = [start, *start.parents]
PROJECT_ROOT = next((p for p in candidates if (p / 'src/project_paths.py').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Start Jupyter from this repository or one of its notebook directories.')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from project_paths import NotebookPaths
paths = NotebookPaths(PROJECT_ROOT, output_group='preprocessing')
input_path, input_glob, output_path = paths.input_path, paths.input_glob, paths.output_path
ALLOW_TRAINING = False  # Explicitly enable before running model training cells.


In [ ]:
import xarray as xr
import numpy as np
import scipy
import cmocean as cmo
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from glob2 import glob
import dask.array
from swath_rossby_wave import inversion
from tqdm import tqdm
from scipy.interpolate import NearestNDInterpolator
import time
from swath_rossby_wave import skill_matrix, build_h_matrix2, build_hswath_matrix2, inversion, make_error_over_time
import matplotlib.patches as mpatches
from cartopy.mpl.ticker import LatitudeFormatter, LongitudeFormatter
from internal_waves import SpectralDomain, calc_gm_wavenumber_spectra, calc_gm_wavenumber_spectra_on_domain, make_synthetic_field, abel_integral


In [ ]:
day0, day1 = 0, 200
n_waves = '190' #number of waves
day0_array = np.arange(63, 63 + 252 * 40, 252)

MModes = 1 # Rossby wave vertical modes
wave_files = input_glob('./new_testing_data_rossby_wave_estimate_*_' + n_waves +'waves_swotdomain_'+ str(int((day1 - day0))) +'days.nc')
wave_files = sorted(wave_files)

lonidx_west, lonidx_east  =  76, 112
latidx_south, latidx_north = 27, 67

new_data = xr.open_dataset(input_path('./aviso_tot_MSLA_ccs_data.nc'))
dsave = new_data.dsave.values
tsave = new_data.tsave.values
xsave = new_data.xsave.values
ysave = new_data.ysave.values
# read in AVISO/Copernicus SSHA data and use to set mask
dsave_transpose = np.transpose(dsave, (1, 0, 2))
SSHA = dsave_transpose[latidx_south:latidx_north, lonidx_west:lonidx_east, :]
T_time = tsave * 86400 # in seconds
T_time = T_time.flatten()
lon, lat = (360 - xsave[lonidx_west:lonidx_east]) * -1, ysave[latidx_south:latidx_north]
dlon = lon - lon.mean()
dlat = lat - lat.mean()
tsave_flat = tsave.flatten()
date_time_all = np.array([np.datetime64(int(atime - tsave_flat[0] + 8401), 'D') for atime in tsave_flat])
ssha_time_mean = SSHA[:, :, : ].mean(axis = -1) # remove multi-year mean (climatology)
ssha_time_mean_expanded = ssha_time_mean[:, :, np.newaxis]
# remove mean from SSH data to produce anomaly over full analysis period
SSHA = SSHA - ssha_time_mean_expanded
#  alternately could remove 80-day mean  SSHA[day0 + day0 + 30].mean(axis = -1)
#  this is not recommended
SSHA_masked = np.ma.masked_invalid(SSHA)
ssha_mask = np.ma.getmask(SSHA_masked)

# set parameters for Rossby wave propagationn model
Phi0 = lat.mean() # central latitude (φ0)
Omega = 7.27e-5 # Ω is the angular speed of the earth
Earth_radius = 6.371e6 / 1e5 # meters
Beta = 2 * Omega * np.cos(Phi0*np.pi/180.) / Earth_radius
f0 = 2 * Omega * np.sin(Phi0*np.pi/180.) #1.0313e-4 # 45 N


In [ ]:
ds = xr.open_dataset(input_path("new_H_swath_test.nc"))
H_swath = ds["H_swath_test"].values


In [ ]:
swot_files = sorted(input_glob('./SWOT_L2_LR_SSH_Expert_474*.nc'))
swot_ds = xr.open_mfdataset(swot_files, combine='nested', concat_dim = 'num_lines') # , engine='store', chunks={'time': 10})
latitude = swot_ds['latitude'].values
longitude = swot_ds['longitude'].values-360

use2=[2, 10, 18, 26,42,50,58,66]
mask=np.zeros([latitude.shape[0],latitude.shape[1]])
for i in range(0,latitude.shape[0]-1,16):
    for j in use2:
        if(latitude[i,j]>=min(lat) and latitude[i,j]<=max(lat)
           and longitude[i,j]<=max(lon) and longitude[i,j]>=min(lon)):
            mask[i,j]=1

index=np.where(mask==1)


In [ ]:
ssh_projection=np.zeros((200,553,59))
MSLA0 = SSHA_masked[:, :, day0:day1] #AVISO input
from tqdm import tqdm
with tqdm(total=len(wave_files)) as pbar:
    for n in range(len(wave_files)):
        wave_ds = xr.open_dataset(input_path(wave_files[n]))  # forward model - filtered AVISO
        # map "true" data on swath
        amp = wave_ds.Amplitudes.data
        ssh = np.matmul(H_swath, amp)
        N = len(ssh)
        ssh_swath = ssh.reshape([MSLA0.shape[2], len(index[0])])
        ssh_projection[:,:,n]=ssh_swath
        pbar.update(1)  # Move this inside the loop to update progress


In [ ]:
# fig, ax = plt.subplots(1, 2, figsize=(13.5, 11.5), subplot_kw={'projection': ccrs.PlateCarree()})
i = 0
fig, ax = plt.subplots(1, 2, figsize=(13.5, 11.5))

index0_half = index[0][:len(index[0])//2]
index1_half = index[1][:len(index[1])//2]
index0_half_rest = index[0][len(index[0])//2:]
index1_half_rest = index[1][len(index[1])//2:]

ssh_swath = ssh.reshape([MSLA0.shape[2], len(index[0])])

sc = ax[0].scatter(longitude[index0_half, index1_half], latitude[index0_half, index1_half], marker='.', c=ssh_swath[i, :len(index0_half)], vmin=-0.2, vmax=0.2, cmap=cmo.cm.balance)
plt.colorbar(sc, ax=ax[0])

ax[0].axis([-128, max(lon), min(lat), max(lat)])
ax[0].set_yticks(np.arange(31.5, 42, 1.5))
# thumbnail_axes('ascending projection', lat, lon, ax[0])

sc2 = ax[1].scatter(longitude[index0_half_rest, index1_half_rest], latitude[index0_half_rest, index1_half_rest], marker='.', c=ssh_swath[i, len(index0_half):], vmin=-0.2, vmax=0.2, cmap=cmo.cm.balance)
plt.colorbar(sc2, ax=ax[1])

ax[1].axis([-128, max(lon), min(lat), max(lat)])
ax[1].set_yticks(np.arange(31.5, 42, 1.5))
# thumbnail_axes('descending projection', lat, lon, ax[1])

plt.show()


In [ ]:
days_of_prediction=[0,10,21,31,42,52,63,73,84,94,105,115,126,136,147,157,168,178,189,199]
ssh_rossby=np.zeros((20,553,59))
for i in range(len(wave_files)):
    for j in range(len(days_of_prediction)):
        ssh_rossby[j,:,i]=ssh_projection[days_of_prediction[j],:,i]


In [ ]:
ds = xr.open_dataset(input_path("new_internal_waves_test.nc"))
ssh_iw = ds["internal_waves_test"].values
lon_i = ds["lon_i"].values
lat_i = ds["lat_i"].values


In [ ]:
#ssh_iw.shape[2]
# Flatten the data matrix
values = ssh_iw[:,:,0].ravel()

# Create the grid for known data points
x = np.linspace(min(lon_i[0,:]), max(lon_i[0,:]), 256)
y = np.linspace(min(lat_i[:,0]), max(lat_i[:,0]), 256)
xv, yv = np.meshgrid(x, y)
points = np.column_stack([xv.ravel(), yv.ravel()])

# Create new points for interpolation
new_points_x = longitude[index[0], index[1]]
new_points_y = latitude[index[0], index[1]]
new_points = np.vstack((new_points_x, new_points_y)).T

# Calculate the time taken for interpolation
start_time = time.time()
interpolator = NearestNDInterpolator(points, values)
interpolated_values = interpolator(new_points)
end_time = time.time()

# Plot the interpolated values
plt.figure(figsize=(13.5, 11.5))
sc3 = plt.scatter(new_points_x, new_points_y, c=interpolated_values, cmap=cmo.cm.balance)
plt.colorbar(sc3)
plt.show()

# Calculate elapsed time
elapsed_time = end_time - start_time
print(f"Time taken for interpolation: {elapsed_time:.4f} seconds")


In [ ]:
ssh_internal=np.zeros((553,1180))
# Create the grid for known data points
x = np.linspace(min(lon_i[0,:]), max(lon_i[0,:]), 256)
y = np.linspace(min(lat_i[:,0]), max(lat_i[:,0]), 256)
xv, yv = np.meshgrid(x, y)
points = np.column_stack([xv.ravel(), yv.ravel()])
# Create new points for interpolation
new_points_x = longitude[index[0], index[1]]
new_points_y = latitude[index[0], index[1]]
new_points = np.vstack((new_points_x, new_points_y)).T

# Interpolation with a loop
for i in range(ssh_iw.shape[2]):
    # Flatten the data matrix
    values = ssh_iw[:,:,i].ravel()
    interpolator = NearestNDInterpolator(points, values)
    interpolated_values = interpolator(new_points)
    ssh_internal[:,i] = interpolated_values


In [ ]:
# Add up Rossby waves and Internal waves
ssh_internal = ssh_internal.reshape(553,20,59)
ssh_rossby = ssh_rossby.transpose(1, 0, 2)
ssh_total= ssh_internal + ssh_rossby


In [ ]:
## Split into ascending and descending
ssh_testing_data = np.zeros((272,20,59))
for i in range(len(wave_files)):
    for j in range(len(days_of_prediction)):
        if j%2==0:
            ssh_testing_data[:,j,i] = ssh_total[:272,j,i]
        else:
            ssh_testing_data[:,j,i] = ssh_total[279:551,j,i]
ssh_testing_data2D = ssh_testing_data.reshape((8,34,20,59))


In [ ]:
ssh_testing_target = np.zeros((272,20,59))
for i in range(len(wave_files)):
    for j in range(len(days_of_prediction)):
        if j%2==0:
            ssh_testing_target[:,j,i] = ssh_rossby[:272,j,i]
        else:
            ssh_testing_target[:,j,i] = ssh_rossby[279:551,j,i]
ssh_testing_target2D = ssh_testing_target.reshape((8,34,20,59))


In [ ]:
ds = xr.Dataset(
    {
        "ssh_testing_data": (["x", "y", "days", "sets"], ssh_testing_data2D),
        "ssh_testing_target": (["x", "y", "days", "sets"], ssh_testing_target2D)
    }
)

# Save the Dataset to a NetCDF file
ds.to_netcdf(output_path("new2_ssh_testing_data2D.nc"))
